# multi_strat_044

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (58).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 4 |
| Detected functions | — |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/ipynb/HVStocks/RELIANCE_minute.csv')
df

In [ ]:
!pip install hmmlearn

In [ ]:
import pandas as pd
import numpy as np
from hmmlearn.hmm import GaussianHMM
import matplotlib.pyplot as plt

# Load and preprocess data
df['date'] = pd.to_datetime(df['date'])
df.sort_values('date', inplace=True)
df['returns'] = np.log(df['close'] / df['close'].shift(1))  # Log returns
df.dropna(inplace=True)

# Feature engineering
df['volatility'] = df['returns'].rolling(window=60).std()  # 20-day rolling volatility
features = df[['returns', 'volatility']].dropna().values

# Train HMM
hmm = GaussianHMM(n_components=3, covariance_type='diag', n_iter=1000)
hmm.fit(features)
df['state'] = np.append([np.nan] * (len(df) - len(features)), hmm.predict(features))

# Set parameters
initial_capital = 100000  # Initial capital
brokerage = 0.001  # 0.1% brokerage
take_profit = 0.02  # 2% take profit
stop_loss = 0.005   # 1% stop loss

# Backtest variables
capital = initial_capital
position = 0
entry_price = 0
cumulative_pnl = 0
tradebook = []

# Backtest
for i in range(len(df)):
    row = df.iloc[i]
    current_date = row['date'].date()
    current_time = row['date'].time()
    state = row['state']
    close_price = row['close']
    high_price = row['high']
    low_price = row['low']

    # Close all positions at 15:28
    if current_time >= pd.Timestamp("15:28:00").time() and position > 0:
        pnl = (close_price - entry_price) * position - brokerage * close_price * position
        capital += pnl
        cumulative_pnl += pnl
        tradebook.append({
            'Entry Time': entry_time,
            'Exit Time': row['date'],
            'Entry Price': entry_price,
            'Exit Price': close_price,
            'PnL': pnl,
            'Action': 'Sell',
            'Cumulative PnL': cumulative_pnl
        })
        position = 0
        entry_price = 0

    # Determine trading action
    if position == 0 and current_time < pd.Timestamp("15:28:00").time():
        if state == 0:  # Bullish state: Buy
            position = initial_capital / close_price
            entry_price = close_price
            entry_time = row['date']
            tradebook.append({
                'Entry Time': entry_time,
                'Exit Time': None,
                'Entry Price': entry_price,
                'Exit Price': None,
                'PnL': None,
                'Action': 'Buy',
                'Cumulative PnL': cumulative_pnl
            })

    # Check take-profit or stop-loss
    if position > 0:
        pnl_percentage = (close_price - entry_price) / entry_price
        if pnl_percentage >= take_profit or pnl_percentage <= -stop_loss:
            pnl = (close_price - entry_price) * position - brokerage * close_price * position
            capital += pnl
            cumulative_pnl += pnl
            tradebook[-1]['Exit Time'] = row['date']
            tradebook[-1]['Exit Price'] = close_price
            tradebook[-1]['PnL'] = pnl
            tradebook[-1]['Action'] = 'Sell'
            tradebook[-1]['Cumulative PnL'] = cumulative_pnl
            position = 0
            entry_price = 0

# Convert tradebook to DataFrame
tradebook_df = pd.DataFrame(tradebook)

# Save tradebook to CSV
tradebook_df.to_csv('tradebook.csv', index=False)

# Summary
print(f"Final Capital: {capital:.2f}")
print(f"Cumulative PnL: {cumulative_pnl:.2f}")


In [ ]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd

# Ensure 'date' is in datetime format and 'close' is numeric
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['close'] = pd.to_numeric(df['close'], errors='coerce')

# Filter out rows with invalid 'date' or 'close' values
df = df.dropna(subset=['date', 'close'])

# Slice the data
df = df.iloc[1:1000]

# Create figure
fig = go.Figure()

# Plot close price
fig.add_trace(go.Scatter(x=df['date'], y=df['close'], mode='lines', name='Close Price'))

# Plot HMM states as different color bands (state 0, 1, 2)
for state in range(3):
    state_data = df[df['state'] == state]
    if not state_data.empty:
        fig.add_trace(go.Scatter(
            x=state_data['date'],
            y=state_data['close'],
            mode='markers',
            name=f"State {state}",
            marker=dict(size=6, color=np.random.choice(['red', 'blue', 'green']), line=dict(width=1))
        ))

# Mark entry and exit points
for trade in tradebook:
    if trade.get('Action') == 'Buy' and 'Entry Time' in trade and 'Entry Price' in trade:
        fig.add_trace(go.Scatter(
            x=[trade['Entry Time']],
            y=[trade['Entry Price']],
            mode='markers',
            name='Buy Signal',
            marker=dict(color='green', size=10, symbol='triangle-up')
        ))
    if trade.get('Action') == 'Sell' and 'Exit Time' in trade and 'Exit Price' in trade:
        fig.add_trace(go.Scatter(
            x=[trade['Exit Time']],
            y=[trade['Exit Price']],
            mode='markers',
            name='Sell Signal',
            marker=dict(color='red', size=10, symbol='triangle-down')
        ))

# Show take-profit and stop-loss levels for the first open trade
if len(tradebook) > 0 and tradebook[0].get('Action') == 'Buy':
    first_trade = tradebook[0]
    if 'Entry Price' in first_trade:
        tp_level = first_trade['Entry Price'] * (1 + take_profit)
        sl_level = first_trade['Entry Price'] * (1 - stop_loss)

        # Plot take-profit and stop-loss levels
        fig.add_trace(go.Scatter(
            x=[first_trade['Entry Time'], first_trade['Exit Time']],
            y=[tp_level, tp_level],
            mode='lines',
            name='Take Profit',
            line=dict(dash='dash', color='green')
        ))
        fig.add_trace(go.Scatter(
            x=[first_trade['Entry Time'], first_trade['Exit Time']],
            y=[sl_level, sl_level],
            mode='lines',
            name='Stop Loss',
            line=dict(dash='dash', color='red')
        ))

# Update layout
fig.update_layout(
    title="HMM Trading Strategy",
    xaxis_title="Date",
    yaxis_title="Price",
    template="plotly_dark",
    showlegend=True
)

# Show plot
fig.show()
